`rich.console.Console.use_theme`
==============================================================================

How does it work inside `__rich_console__`?

This is the "standard" way of using `Console.use_theme` that I've seen, directly
coupled with `Console.print` statements.

As you can see in the output, printing inside the `use_theme` uses the `t_2`
style.

In [9]:
from rich.console import (
    Console,
    ConsoleOptions,
    RenderResult,
    ConsoleRenderable,
)
from rich.style import Style
from rich.text import Text
from rich.theme import Theme
from rich.panel import Panel
from rich.pretty import install
from rich.traceback import Traceback

import splatlog as slog

slog.setup(
    level=slog.INFO,
    console=True,
    theme=slog.rich.THEME_ANSI_DARK,
)

install()

t_1 = Theme({"x": Style(color="red")})
t_2 = Theme({"x": Style(color="cyan")})

text = Text("hey", style="x")
c = Console(theme=t_1, width=80)

c.print(text)

with c.use_theme(t_2):
    c.print(text)

hey

hey

So this was the question: does it still work when yielding renderables inside `__rich_console__`?

In [10]:
class A:
    def __rich_console__(
        self, console: Console, options: ConsoleOptions
    ) -> RenderResult:
        yield Text("using console theme", style="x")

        with console.use_theme(t_2):
            yield Text("using t_2", style="x")

        yield Text("back to console theme", style="x")


c.print(A())

using console theme
using t_2
back to console theme

As you can see in the above output, the answer is affirmative: the `t_2` style
is applied to objects yielded within (and only within) the `use_theme` block.

This appears to work regardless of where the renderables are created; even a
`Text` created outside the `use_theme` block that is yielded from inside it
adopts the used `Theme`, as demonstrated below:

In [11]:
from rich.console import Group


def render_panel(title: str, *content: ConsoleRenderable) -> ConsoleRenderable:
    return Panel(
        Group(Text("in function", style="x"), *content),
        title=title,
    )


class B:
    def __rich_console__(
        self, console: Console, options: ConsoleOptions
    ) -> RenderResult:
        outside = Text("outside", style="x")
        yield render_panel("using console theme", outside)

        with console.use_theme(t_2):
            inside = Text("inside", style="x")
            yield render_panel("using t_2", outside, inside)

        yield render_panel("back to console theme", outside)


c.print(B())

╭──────────────────────────── using console theme ─────────────────────────────╮
│ in function                                                                  │
│ outside                                                                      │
╰──────────────────────────────────────────────────────────────────────────────╯
╭───────────────────────────────── using t_2 ──────────────────────────────────╮
│ in function                                                                  │
│ outside                                                                      │
│ inside                                                                       │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────── back to console theme ────────────────────────────╮
│ in function                                                                  │
│ outside                                                                      │
╰──────────────────────────────────────────────────────────────────────────────╯

In [12]:
from rich.markdown import Markdown


def get_value_error() -> ValueError:
    """Create a ValueError with interesting local variables."""
    try:
        some_list = [1, 2, 3]
        some_dict = {"key": "value", "count": 42}
        some_string = "hello world"
        raise ValueError(f"Invalid data: {some_list}, flag={True}, count={123}")
    except ValueError as err:
        return err


val_err = get_value_error()
tb = Traceback.from_exception(
    type(val_err),
    val_err,
    val_err.__traceback__,
    show_locals=True,
    theme="github-dark",
)
tb_theme = slog.rich.to_theme(tb.theme)

Markdown("\n".join(f"-  `{k}` — {v}" for k, v in tb_theme.styles.items()))

 • pretty — not bold not italic not underline #e6edf3 on #0d1117                                                   
 • pygments.text — not bold not italic not underline #e6edf3 on #0d1117                                            
 • pygments.string — not bold not italic not underline #a5d6ff on #0d1117                                          
 • pygments.function — bold not italic not underline #d2a8ff on #0d1117                                            
 • pygments.number — not bold not italic not underline #a5d6ff on #0d1117                                          
 • repr.indent — not bold dim italic not underline #8b949e on #0d1117                                              
 • repr.str — not bold not italic not underline #a5d6ff on #0d1117                                                 
 • repr.brace — bold not italic not underline #e6edf3 on #0d1117                                                   
 • repr.number — not bold not italic not underline #a5d6ff on #0d1117                                              
 • repr.bool_true — not bold not italic not underline #79c0ff on #0d1117                                           
 • repr.bool_false — not bold not italic not underline #79c0ff on #0d1117                                          
 • repr.none — not bold not italic not underline #79c0ff on #0d1117                                                
 • scope.border — not bold not italic not underline #79c0ff on #0d1117                                             
 • scope.equals — bold not italic not underline #ff7b72 on #0d1117                                                 
 • scope.key — not bold not italic not underline #e6edf3 on #0d1117                                                
 • scope.key.special — bold dim not italic not underline #79c0ff on #0d1117

### Printing `rich.traceback.Traceback`

Despite the use of  `Console.use_theme(traceback_theme)` we can see in
`rich.traceback`, printing a `Traceback` does _not_ apply the syntax theme
outside `rich.syntax.Syntax` elements.

From perusing the source code it certainly seems like the intent is to apply the
`rich.syntax.SyntaxTheme` styles to the _Traceback (most recent call last)_
and syntax error panels, but they don't show up.

In [13]:
c.print(tb)

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ in get_value_error:10                                                        │
│                                                                              │
│    7 │   │   some_list = [1, 2, 3]                                           │
│    8 │   │   some_dict = {"key": "value", "count": 42}                       │
│    9 │   │   some_string = "hello world"                                     │
│ ❱ 10 │   │   raise ValueError(f"Invalid data: {some_list}, flag={True}, coun │
│   11 │   except ValueError as err:                                           │
│   12 │   │   return err                                                      │
│   13                                                                         │
│                                                                              │
│ ╭────────────────── locals ───────────────────╮                              │
│ │   some_dict = {'key': 'value', 'count': 42} │                              │
│ │   some_list = [1, 2, 3]                     │                              │
│ │ some_string = 'hello world'                 │                              │
│ ╰─────────────────────────────────────────────╯                              │
╰──────────────────────────────────────────────────────────────────────────────╯
ValueError: Invalid data: [1, 2, 3], flag=True, count=123

If we instead wrap the `Console.print` in `use_theme` we see the `SyntaxTheme`
styles get applied; notice the highlighting on `get_value_error` and in the
_local_ panel, both of which reside in the _Traceback_ panel.

In [14]:
with c.use_theme(tb_theme):
    c.print(tb)

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ in get_value_error:10                                                        │
│                                                                              │
│    7 │   │   some_list = [1, 2, 3]                                           │
│    8 │   │   some_dict = {"key": "value", "count": 42}                       │
│    9 │   │   some_string = "hello world"                                     │
│ ❱ 10 │   │   raise ValueError(f"Invalid data: {some_list}, flag={True}, coun │
│   11 │   except ValueError as err:                                           │
│   12 │   │   return err                                                      │
│   13                                                                         │
│                                                                              │
│ ╭────────────────── locals ───────────────────╮                              │
│ │   some_dict = {'key': 'value', 'count': 42} │                              │
│ │   some_list = [1, 2, 3]                     │                              │
│ │ some_string = 'hello world'                 │                              │
│ ╰─────────────────────────────────────────────╯                              │
╰──────────────────────────────────────────────────────────────────────────────╯
ValueError: Invalid data: [1, 2, 3], flag=True, count=123

In [15]:
c.print(slog.rich.enrich(val_err))

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ in get_value_error:10                                                        │
│                                                                              │
│    7 │   │   some_list = [1, 2, 3]                                           │
│    8 │   │   some_dict = {"key": "value", "count": 42}                       │
│    9 │   │   some_string = "hello world"                                     │
│ ❱ 10 │   │   raise ValueError(f"Invalid data: {some_list}, flag={True}, coun │
│   11 │   except ValueError as err:                                           │
│   12 │   │   return err                                                      │
│   13                                                                         │
╰──────────────────────────────────────────────────────────────────────────────╯
ValueError: Invalid data: [1, 2, 3], flag=True, count=123